# Dependencies

In [28]:
# Deep Learning
from keras.layers import Input, Flatten
from keras.layers import Conv2D, Dense, Activation, BatchNormalization
from keras.layers import MaxPooling2D, GlobalAveragePooling2D
from keras.models import Model

# Linear Algebra
import numpy as np

# IO
import glob

# Visualization
from matplotlib import pyplot as plt
from matplotlib.pyplot import imread, imshow

# Load the dataset

In [91]:
def shuffle_simultaneously(a, b):
    '''
    Shuffles 2 separate numpy arrays simultaneously
    '''
    # Generate the permutation index array.
    permutation = np.random.permutation(a.shape[0])
    
    # Shuffle the arrays by giving the permutation in the square brackets.
    shuffled_a = a[permutation]
    shuffled_b = b[permutation]
    return shuffled_a, shuffled_b

In [92]:
def load_data(directory):
    '''
    Loads the concrete data from the specified directory
    '''
    # Get filenames
    neg_files = glob.glob(directory + '/Negative/*.jpg')
    pos_files = glob.glob(directory + '/Positive/*.jpg')
    
    # Load negative examples
    neg_imgs = np.array([imread(file) for file in neg_files[:100]])
    neg_lbls = np.zeros([len(neg_imgs), 1])
    
    # Load positive examples
    pos_imgs = np.array([imread(file) for file in pos_files[:100]])
    pos_lbls = np.ones([len(pos_imgs), 1])
    
    ### Train / Val / Test split
    size = len(neg_imgs)
    
    # Negative training examples
    X_train_neg = neg_imgs[:int(size * 0.7)]
    y_train_neg = neg_lbls[:int(size * 0.7)]
    
    # Negative validation examples
    X_val_neg = neg_imgs[int(size * 0.7) : int(size * 0.85)]
    y_val_neg = neg_lbls[int(size * 0.7) : int(size * 0.85)]
    
    # Negative testing examples
    X_test_neg = neg_imgs[int(size * 0.85):]
    y_test_neg = neg_lbls[int(size * 0.85):]
    
    # Positive training examples
    X_train_pos = pos_imgs[:int(size * 0.7)]
    y_train_pos = pos_lbls[:int(size * 0.7)]

    # Positive validation examples
    X_val_pos = pos_imgs[int(size * 0.7) : int(size * 0.85)]
    y_val_pos = pos_lbls[int(size * 0.7) : int(size * 0.85)]
    
    # Positive testing examples
    X_test_pos = pos_imgs[int(size * 0.85):]
    y_test_pos = pos_lbls[int(size * 0.85):]
    
    # Training set
    X_train = np.concatenate([X_train_neg, X_train_pos])
    y_train = np.concatenate([y_train_neg, y_train_pos])
    
    # Validation set
    X_val = np.concatenate([X_val_neg, X_val_pos])
    y_val = np.concatenate([y_val_neg, y_val_pos])
    
    # Test set
    X_test = np.concatenate([X_test_neg, X_test_pos])
    y_test = np.concatenate([y_test_neg, y_test_pos])
    
    # Shuffle datasets
    X_train, y_train = shuffle_simultaneously(X_train, y_train)
    X_val, y_val = shuffle_simultaneously(X_val, y_val)
    X_test, y_test = shuffle_simultaneously(X_test, y_test)

    return X_train, y_train, X_val, y_val, X_test, y_test

In [85]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data('./data')

# Define a Neural Network

In [93]:
def NeuralNetwork():
    '''
    Small convolutional neural network for sigmoidal image classification
    '''
    # Model input
    input_img = Input(shape=(227,227,3))
    
    # Conv 1
    x = Conv2D(filters=32, kernel_size=(3,3), kernel_initializer='he_uniform')(input_img)
    x = BatchNormalization()(x)
    x = Activation(activation='relu')(x)
    x = MaxPooling2D()(x)
    
    # Conv 2
    x = Conv2D(filters=64, kernel_size=(3,3), kernel_initializer='he_uniform')(x)
    x = BatchNormalization()(x)
    x = Activation(activation='relu')(x)
    x = MaxPooling2D()(x)
    
    # Conv 3
    x = Conv2D(filters=128, kernel_size=(3,3), kernel_initializer='he_uniform')(x)
    x = BatchNormalization()(x)
    x = Activation(activation='relu')(x)
    x = MaxPooling2D()(x)
    
    # Flatten convolutional output for input to fully connected layers
    x = Flatten()(x)
    
    # Fully connected layer 
    x = Dense(512, kernel_initializer="he_uniform")(x)
    x = Activation(activation='relu')(x)
    
    # Logistic layer
    output = Dense(1, activation="sigmoid")(x)
    
    # Create model
    model = Model(inputs=[input_img], outputs=[output])
    
    model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['acc'])
    
    return model

In [94]:
model = NeuralNetwork()
model.summary()

_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_3 (InputLayer)         (None, 227, 227, 3)       0         
_________________________________________________________________
conv2d_7 (Conv2D)            (None, 225, 225, 32)      896       
_________________________________________________________________
batch_normalization_7 (Batch (None, 225, 225, 32)      128       
_________________________________________________________________
activation_9 (Activation)    (None, 225, 225, 32)      0         
_________________________________________________________________
max_pooling2d_7 (MaxPooling2 (None, 112, 112, 32)      0         
_________________________________________________________________
conv2d_8 (Conv2D)            (None, 110, 110, 64)      18496     
_________________________________________________________________
batch_normalization_8 (Batch (None, 110, 110, 64)      256       
__________

# Model Training

In [ ]:
history = model.fit(x=X_train, 
                    y=y_train, 
                    batch_size=64, 
                    epochs=150, 
                    validation_data=(X_val, y_val), 
                    shuffle=True)

In [ ]:
# Plot training & validation accuracy values
plt.plot(history.history['acc'])
plt.plot(history.history['val_acc'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Test'], loc='upper left')
plt.show()

# Evaluation

In [ ]:
model.evaluate(x=X_test, y=y_test)

# Visualize the Class Activation Map